# Практика 4. Чищення набору даних

**Студент:** Yurii_huz  
**Група:** IT-41  
**Варіант:** 1, Київ  
**Базова ціна:** 850 грн  
**Базова кількість:** 3

Мета: побудувати брудний набір даних, знайти пропуски, дублікати, неправильні типи, неузгоджені категорії та викид, а потім послідовно усунути ці дефекти.

In [ ]:
import numpy as np
import pandas as pd

## Завдання 1. Побудова брудного набору

In [ ]:
dirty = pd.DataFrame([
    {"номер_замовлення": 1, "дата": "2026-09-01", "товар": "термочашка", "місто": "Київ", "ціна": 850, "кількість": 2},
    {"номер_замовлення": 2, "дата": "2026-09-02", "товар": "термочашка", "місто": " Київ ", "ціна": "850 грн", "кількість": np.nan},
    {"номер_замовлення": 3, "дата": "2026-09-03", "товар": "термочашка", "місто": "КИЇВ", "ціна": 850, "кількість": 5},
    {"номер_замовлення": 4, "дата": "2026-09-04", "товар": "термочашка", "місто": "Київ", "ціна": "850 грн", "кількість": 3},
    {"номер_замовлення": 5, "дата": "2026-09-05", "товар": "термочашка", "місто": "КИЇВ", "ціна": 850, "кількість": np.nan},
    {"номер_замовлення": 6, "дата": "2026-09-06", "товар": "термочашка", "місто": "Київ", "ціна": 8500, "кількість": 1},
    {"номер_замовлення": 7, "дата": "2026-09-07", "товар": "термочашка", "місто": "Київ", "ціна": 850, "кількість": 4},
    {"номер_замовлення": 7, "дата": "2026-09-07", "товар": "термочашка", "місто": "Київ", "ціна": 850, "кількість": 4},
])

print(dirty)
print()
print("Кількість рядків:", len(dirty))
print("Пропуски за стовпцями:")
print(dirty.isna().sum())
print("Повні дублікати:", dirty.duplicated().sum())
print("Унікальні варіанти міста:", dirty["місто"].unique())
print("Типи ціни до очищення:", dirty["ціна"].map(type).unique())

У наборі є 8 рядків, два пропуски в `кількість`, один повний дублікат, два значення ціни з суфіксом `грн`, три написання міста (`Київ`, ` Київ `, `КИЇВ`) і викид `8500 грн`, що у 10 разів більший за базову ціну.

## Завдання 2. Пропуски

In [ ]:
missing_before = dirty.isna().sum()
print("Кількість пропусків до заповнення:")
print(missing_before)

quantity_median = dirty["кількість"].median()
dirty["кількість"] = dirty["кількість"].fillna(quantity_median)
print("Медіана кількості:", quantity_median)
print("Пропуски після заповнення:")
print(dirty.isna().sum())

Медіана доречніша за середнє, тому що в наборі лише 8 спостережень і є аномально велика ціна. Медіана менш чутлива до крайніх значень та краще представляє типову кількість замовлення.

## Завдання 3. Дублікати

In [ ]:
full_duplicates = dirty.duplicated()
order_duplicates = dirty.duplicated(subset=["номер_замовлення"])
print("Повні дублікати:")
print(dirty[full_duplicates])
print("Дублікати за номером замовлення:")
print(dirty[order_duplicates])

dirty = dirty.drop_duplicates(subset=["номер_замовлення"]).copy()
print("Кількість рядків після видалення дубліката:", len(dirty))

## Завдання 4. Типи даних і узгодження категорій

In [ ]:
dirty["ціна"] = (
    dirty["ціна"]
    .astype(str)
    .str.replace(" грн", "", regex=False)
    .astype(float)
)

dirty["місто"] = (
    dirty["місто"]
    .str.strip()
    .str.lower()
    .replace({"київ": "Київ"})
)

print("Тип ціни після очищення:", dirty["ціна"].dtype)
print("Унікальні назви міста після очищення:", dirty["місто"].unique())
print(dirty)

## Завдання 5. Викиди

In [ ]:
q1, q3 = dirty["ціна"].quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = dirty[(dirty["ціна"] < lower) | (dirty["ціна"] > upper)]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Межі IQR:", (lower, upper))
print("Значення поза межами IQR:")
print(outliers[["номер_замовлення", "ціна"]])

Значення `8500 грн` виходить за межі IQR, оскільки більшість замовлень мають ціну `850 грн`, а верхня межа дорівнює `850 грн`. Для цього навчального набору я вважаю `8500 грн` помилкою вводу із зайвим нулем і замінив би його на `850 грн`. У реальному магазині перед заміною потрібно перевірити первинне замовлення: автоматичне видалення могло б прибрати справді велике замовлення.

In [ ]:
clean = dirty.copy()
clean.loc[clean["ціна"] == 8500, "ціна"] = 850
print("Очищений набір:")
print(clean)

## Письмові відповіді

1. Заповнення пропуску середнім може спотворити стандартне відхилення, бо середнє залежить від усіх значень і крайніх спостережень. Крім того, підставлені середні значення штучно концентрують дані біля центру та зменшують оцінену варіативність.

2. `duplicated()` без `subset` шукає повністю однакові рядки за всіма стовпцями. `duplicated(subset=["номер_замовлення"])` перевіряє дублювання лише за вказаним полем, тому може знайти рядки, які відрізняються іншими значеннями.

3. Автоматичне видалення всіх значень поза межами IQR не завжди правильне, бо викид може бути не помилкою, а рідкісною, але реальною подією. Перед видаленням потрібно перевірити джерело даних і предметну область.

## Підсумок

У наборі Києва створено та послідовно усунуто пропуски, дублікат, неправильні типи ціни, різні варіанти назви міста й викид. Межі IQR використано як формальний сигнал, а остаточне рішення щодо викиду обґрунтовано окремо.